## Task 1 (Entrega Parcial)


### Task 1.1

Antes de implementar cualquier algoritmo, el equipo de ingeniería necesita una justificación técnica sólida
de las decisiones de diseño.


a. El espacio de acción del sistema de control de palas es continuo. Argumenten formalmente por qué DQN y sus variantes de la semana anterior son inapropiados para este dominio. Su argumento debe mencionar explícitamente el operador argmax, el espacio de acción continuo, y las consecuencias computacionales de intentar aplicar DQN directamente.


DQN está diseñado para un conjunto de acciones discreto y finito. Su objetivo TD utiliza el target

$$y_t=r_t+\gamma(1-d_t)\max_{a'\in\mathcal A}Q_{\theta^-}(s_{t+1},a'),$$

donde la política implícita es $a^*(s)=\arg\max_{a\in\mathcal A}Q(s,a)$. El operador $\arg\max$ puede evaluarse enumerando las acciones porque DQN produce un valor de salida por cada acción discreta.

En el control de palas, $\mathcal A$ es continuo, por ejemplo un intervalo o un subconjunto de $\mathbb R^m$. Por tanto, hay infinitas acciones y una red DQN no puede tener una salida por cada una. Aplicar directamente el mismo operador exigiría resolver, para cada siguiente estado y para cada actualización, el problema de optimización continua $\max_{a'}Q(s_{t+1},a')$. Esa optimización sería costosa, posiblemente no convexa y tendría que repetirse millones de veces durante el entrenamiento; además, el gradiente del `argmax` no es adecuado para el entrenamiento estándar de DQN.

Discretizar el intervalo tampoco elimina el problema: una discretización fina produce muchas acciones y, si hay varias palas o actuadores, el número de combinaciones crece exponencialmente con la dimensión. Una discretización gruesa introduce error de cuantización y puede generar comandos físicamente inadecuados. Las variantes de DQN conservan esta dependencia del máximo sobre acciones discretas, por lo que no son la elección apropiada. Algoritmos actor--critic como TD3 y SAC parametrizan directamente una acción continua y evitan enumerar todo el espacio.


b. Entre TD3 y SAC, ¿cuál recomendarían para el sistema de control de turbinas eólicas en producción? Justifiquen considerando: la naturaleza física del sistema (¿es deseable una política estocástica en un sistema de control industrial?), la eficiencia de datos durante el entrenamiento en simulación, y la robustez ante perturbaciones de viento no vistas durante el entrenamiento. Su respuesta debe argumentar ambos lados antes de tomar posición.


Ambos métodos son off-policy y reutilizan el replay buffer, por lo que son mucho más eficientes en datos que un método on-policy. TD3 aprende una política determinista $\mu_\theta(s)$ y usa dos críticos, el mínimo entre ellos, actualización retardada del actor y suavizado del target. Esto reduce el sobreestimado de $Q$ y produce comandos repetibles, una propiedad valiosa cuando una variación aleatoria del pitch o del torque puede aumentar cargas estructurales, fatiga o desgaste del actuador. En un simulador suave y suficientemente fiel, TD3 suele ser una opción muy eficiente y directa para control continuo.

SAC, por otro lado, optimiza recompensa más entropía. Su política Gaussiana reparametrizada explora de forma explícita y el ajuste automático de $\alpha$ evita fijar manualmente el nivel de exploración. Esa diversidad de acciones puede ayudar a escapar de soluciones locales y a aprender una política menos frágil frente a cambios de turbulencia, densidad del aire o velocidad del viento que no estaban presentes en el entrenamiento. También puede ser ventajosa si el simulador tiene incertidumbre y se desea que la política aprenda varias acciones razonables en un mismo estado.

La desventaja de SAC en producción es que su política es estocástica: la aleatoriedad útil durante entrenamiento no debe traducirse sin control a comandos del sistema físico. Además, el término de entropía puede favorecer acciones variadas aunque dos acciones tengan casi la misma recompensa, y su beneficio depende de calibrar correctamente $\alpha$ y de tener un simulador representativo. SAC puede ejecutarse usando la media de la política en despliegue, pero entonces parte de la motivación estocástica desaparece y aún se debe validar el comportamiento fuera de distribución.

**Recomendación:** elegiría **TD3 para el controlador de producción**, acompañado de límites físicos, un supervisor de seguridad y validación con perturbaciones de viento. La determinismo, el suavizado del target y la menor variabilidad de las acciones encajan mejor con un sistema industrial. Usaría SAC como referencia de entrenamiento o como candidato alternativo si los experimentos con turbulencias no vistas muestran una mejora clara y se puede filtrar la acción mediante una capa de seguridad. La decisión debe confirmarse con pruebas de robustez, no solo con la recompensa promedio.


c. La empresa tiene tres años de registros históricos generados por operadores humanos con distintos niveles de experiencia. Analicen si ese dataset es apropiado para aplicar CQL. Su análisis debe considerar: qué tipo de política de comportamiento 𝛽 generó los datos (¿es cercana a la óptima?), qué cobertura del espacio estado-acción se puede esperar de operadores humanos, y cuál de las dos condiciones de Offline RL es más difícil de satisfacer en este dominio específico.


El dataset puede ser apropiado para CQL, pero no garantiza por sí mismo una solución segura ni óptima. La política de comportamiento puede modelarse como una mezcla $\beta(a\mid s)=\sum_i w_i\beta_i(a\mid s)$, donde cada $\beta_i$ corresponde a un operador con diferente experiencia, criterio y tolerancia al riesgo. No es razonable asumir que $\beta$ sea cercana a la política óptima: los operadores suelen priorizar seguridad, estabilidad y protocolos conservadores, y pueden existir trayectorias subóptimas o decisiones inconsistentes. La heterogeneidad sí puede ser útil porque incorpora más de una estrategia y más estados operativos.

La cobertura será buena alrededor de los regímenes normales observados, pero probablemente pobre en combinaciones críticas: ráfagas extremas, fallas, recuperación después de saturación y acciones agresivas que un operador prudente evita probar. También puede haber poca cobertura de estados raros y de acciones continuas entre los comandos usados históricamente. La diversidad de operadores no equivale a exploración uniforme; puede repetir acciones seguras en la misma región del espacio.

Las dos condiciones prácticas de Offline RL son: (1) **cobertura o solapamiento**, es decir, que las acciones que la política aprendida quiera usar tengan soporte suficiente en el dataset; y (2) **calidad suficiente de los datos**, con trayectorias que contengan comportamiento competente y señales para distinguir decisiones buenas de malas. CQL ayuda con el error de extrapolación y puede trabajar con datos subóptimos, pero no puede inventar evidencia fiable para acciones o estados ausentes. En este dominio, la condición más difícil es la **cobertura en estados y acciones críticos**, porque la operación segura evita precisamente experimentar con ellos. Por tanto, aplicaría CQL con análisis de cobertura, separación por régimen de viento y validación en simulación antes de cualquier uso operativo.


d. Argumenten cuál es el riesgo principal de aplicar Q-Learning estándar directamente sobre el dataset histórico sin ninguna restricción de conservatividad. Usen el concepto de error de extrapolación y conéctenlo con una consecuencia operacional concreta para el sistema de turbinas.


El riesgo principal es que el Q-Learning offline seleccione acciones fuera de la distribución del dataset porque el máximo del backup favorece cualquier error positivo del aproximador. Si una acción $(s,a)$ casi nunca aparece, su valor se obtiene por extrapolación de la red; ese error puede ser grande y positivo. El operador $\max_a Q(s,a)$ convierte ese error en una acción elegida por la política y, mediante el bootstrapping, lo propaga a estados anteriores. Es el llamado **error de extrapolación** o distribución fuera de soporte.

En una turbina, el resultado puede ser que el controlador prefiera un ángulo de pala o un torque que parece excelente para el modelo, pero nunca fue validado por los operadores en ese estado de viento. El comando podría saturar el actuador, producir oscilaciones de carga, aumentar la fatiga, llevar el rotor a una condición de sobrevelocidad o provocar una parada de emergencia. La recompensa estimada sería optimista mientras el sistema real sufriría una degradación o una condición insegura; por eso se necesita una penalización conservadora, restricciones de acción y una capa independiente de protección.


### Task 1.2

El equipo quiere entender las diferencias matemáticas entre los algoritmos antes de implementarlos.


a. Escriban el objetivo de optimización de SAC incluyendo el término de entropía. Identifiquen cada variable con su significado en el contexto del sistema de turbinas: ¿qué representa 𝛼, qué representa ℋ(𝜋(⋅∣ 𝑆𝑡)), y cómo se calibraría 𝛼 automáticamente durante el entrenamiento?


**Respuesta.** El objetivo de SAC puede escribirse como

$$J(\pi)=\mathbb E_{\tau\sim\pi}\left[\sum_{t=0}^{\infty}\gamma^t\left(r(s_t,a_t)+\alpha\,\mathcal H(\pi(\cdot\mid s_t))\right)\right],$$

con
$$\mathcal H(\pi(\cdot\mid s))=-\mathbb E_{a\sim\pi(\cdot\mid s)}[\log\pi(a\mid s)].$$

$s_t$ contiene el estado operativo de la turbina (por ejemplo, velocidad del rotor, viento y cargas), $a_t$ es el comando continuo de pitch/torque, $r$ mide desempeño y seguridad, y $\gamma$ descuenta el futuro. $\mathcal H$ mide la incertidumbre o diversidad de acciones que la política mantiene en un estado. $\alpha>0$ es la temperatura de entropía: un valor alto prioriza exploración y tolerancia a varias acciones, mientras que un valor bajo prioriza maximizar la recompensa y hace la política más determinista.

Para ajustar $\alpha$ automáticamente se fija una entropía objetivo $\bar{\mathcal H}$ y se optimiza $\log\alpha$ como variable libre. En la convención usual de SAC para acciones continuas se toma $\bar{\mathcal H}=-\dim(\mathcal A)$, como solicita el enunciado. La pérdida de temperatura es

$$L_{\log\alpha}=\mathbb E_{s\sim D,,a\sim\pi}\left[-\log\alpha\,\bigl(\log\pi(a\mid s)+\bar{\mathcal H}\bigr)\right].$$

La actualización aumenta $\alpha$ cuando la entropía observada está por debajo del objetivo y la reduce cuando está por encima. En control real, la acción usada al evaluar o desplegar debe pasar además por límites, rate limiters y verificaciones de seguridad.


b. Comparen el target de actualización del Critic en TD3 versus SAC. ¿Cuál de los dos produce estimaciones más conservadoras del valor y por qué? ¿Esa conservatividad es una ventaja o una desventaja para el dominio de turbinas?


En TD3, el target del crítico es

$$y_t^{TD3}=r_t+\gamma(1-d_t)\min_{i\in\{1,2\}}Q_{\phi_i^-}\left(s_{t+1},\mu_{\theta^-}(s_{t+1})+\operatorname{clip}(\epsilon,-c,c)\right),$$

donde $\epsilon\sim\mathcal N(0,\sigma^2)$ y la acción se recorta al rango válido. En SAC, usando la misma convención de dos críticos,

$$y_t^{SAC}=r_t+\gamma(1-d_t)\left[\min_{i\in\{1,2\}}Q_{\phi_i^-}(s_{t+1},a_{t+1})-\alpha\log\pi_\theta(a_{t+1}\mid s_{t+1})\right],$$

con $a_{t+1}\sim\pi_\theta(\cdot\mid s_{t+1})$. Ambos usan el mínimo de dos críticos, lo que reduce el sobreestimado. Sin embargo, el término $-\alpha\log\pi$ es un bono de entropía: en promedio añade $\alpha\mathcal H$ al valor soft. Por ello, **en magnitud numérica y manteniendo iguales los críticos, TD3 suele producir el target más conservador**; SAC puede tener un valor soft mayor aunque su mínimo de críticos también controle el optimismo. El suavizado de TD3 añade además tolerancia a pequeños errores alrededor de la acción seleccionada.

En turbinas, cierta conservatividad es una ventaja: reduce la probabilidad de que el controlador elija una acción no validada por una sobreestimación del modelo. El exceso de conservatividad también es una desventaja porque puede producir control tímido, menor captura de energía o rechazo de maniobras útiles. La solución práctica es combinar el crítico conservador con recompensas que penalicen cargas y saturación, calibrar $\alpha$ y validar rendimiento y seguridad bajo perturbaciones.


c. Escriban el objetivo de CQL identificando los dos términos que se añaden a la pérdida TD estándar. Expliquen en palabras qué hace cada término y por qué su combinación produce un Critic conservador. Luego argumenten qué valor de 𝛼𝐶𝑄𝐿 recomendarían para el dataset de turbinas considerando la heterogeneidad de los operadores que lo generaron.


Para acciones continuas, una forma de CQL es

$$L_{CQL}(Q)=L_{TD}(Q)+\alpha_{CQL}\,\mathbb E_{s\sim D}\left[\underbrace{\log\int_{\mathcal A}\exp(Q(s,a))\,da}_{\text{término log-sum-exp}}-\underbrace{\mathbb E_{a\sim D(\cdot\mid s)}Q(s,a)}_{\text{término de acciones del dataset}}\right].$$

En implementación, la integral se aproxima con acciones muestreadas de la política, del prior/uniforme y del dataset. Los dos términos añadidos son, por tanto, (1) el **log-sum-exp**, que penaliza valores altos en cualquier acción considerada, especialmente acciones fuera de distribución; y (2) el término negativo sobre acciones observadas, que evita penalizar de la misma manera las acciones respaldadas por datos y las mantiene como referencia. La combinación reduce $Q$ donde el crítico no tiene evidencia y conserva valores relativamente mayores en el soporte del dataset. Así, la política deja de explotar errores positivos de extrapolación.

Para los registros de operadores heterogéneos recomendaría comenzar con $\alpha_{CQL}=1.0$. Es un compromiso razonable: aporta conservatividad ante acciones poco respaldadas sin borrar por completo las diferencias entre operadores expertos y novatos. Probaría $0.5$ y $5.0$ como análisis de sensibilidad; usaría $5.0$ si la auditoría confirma cobertura muy pobre en regiones críticas y se prioriza seguridad, pero vigilaría el subestimado y el colapso hacia una política excesivamente imitativa. La selección final debe hacerse con validación por régimen de viento, métricas de seguridad y evaluación en simulación, no únicamente con la pérdida CQL.


## Task 2


### Task 2.1

Implementen TD3 y SAC sobre el entorno Pendulum-v1 de Gymnasium como proxy del sistema de control de turbinas. Este entorno tiene espacio de acción continuo unidimensional en [−2,2] y estado tridimensional, lo que lo hace apropiado para validar las propiedades algorítmicas antes de escalar al dominio real.

Ambos algoritmos deben implementarse desde cero usando PyTorch. No se permite usar implementaciones preconstruidas de TD3 o SAC de ninguna librería de RL. Usen las siguientes especificaciones compartidas para ambos: buffer de Experience Replay con capacidad de 100,000 transiciones, minibatch de 256 transiciones, Target Networks con actualización suave 𝜏 = 0.005, dos capas ocultas de 256 neuronas con activación ReLU, y entrenamiento durante 100,000 pasos de ambiente.

Para TD3 específicamente: Twin Critics, Policy Delay de 2 pasos, y Target Policy Smoothing con 𝜎 = 0.2 y clip c = 0.5.

Para SAC específicamente: política Gaussiana reparametrizada con tanh aplicado a la salida para restringir acciones al rango válido, Twin Critics, y 𝛼 ajustado automáticamente con entropía objetivo −dim(𝒜).

La implementación debe registrar por cada 1,000 pasos: recompensa promedio de evaluación greedy sobre 10 episodios, valor Q promedio estimado sobre un conjunto fijo de 1,000 transiciones, y entropía de la política (solo para SAC).


### Task 2.2


Generen el dataset para el experimento de Offline RL entrenando primero una política subóptima sobre Pendulum-v1: entrenen TD3 durante solo 20,000 pasos (sin convergencia completa) y recolecten 50,000 transiciones con esa política más exploración adicional con ruido gaussiano 𝒩(0,0.3). Este dataset simula los registros históricos de operadores humanos con desempeño heterogéneo.

Implementen CQL sobre ese dataset. Usen la misma arquitectura de red que en Task 2.1. La pérdida de CQL debe incluir el término TD estándar más el término de regularización con 𝛼𝐶𝑄𝐿 ∈ {0.5,1.0,5.0}. Entrenen durante 100,000 actualizaciones sobre el dataset sin ninguna interacción adicional con el entorno.

Evalúen la política aprendida por CQL ejecutándola sobre el entorno real durante 10 episodios por cada valor de 𝛼𝐶𝑄𝐿. Comparen esa evaluación con la recompensa promedio de la política subóptima que generó el dataset.


### Task 2.3

Con base en los resultados de ambos experimentos, respondan:


a. Grafiquen las curvas de aprendizaje de TD3 y SAC en la misma figura con media móvil de 10
evaluaciones. ¿Cuál converge más rápido? ¿Cuál alcanza mayor recompensa final? ¿El resultado
coincide con la predicción teórica de la Task 1.1b? Si hay discrepancia, argumenten la causa.


b. Grafiquen el valor Q promedio durante el entrenamiento para TD3 y SAC. ¿Observan diferencias en la magnitud de los valores estimados? Conéctenlo con el análisis de conservatividad de la Task 1.2b.


c. Para el experimento de CQL: ¿alguno de los tres valores de 𝛼𝐶𝑄𝐿 produce una política que supera a la política subóptima que generó el dataset? ¿Cuál valor de 𝛼𝐶𝑄𝐿 funciona mejor y por qué? Si ningún valor supera al dataset, argumenten a qué atribuyen ese resultado considerando las características del dataset que generaron.


d. Conexión entre paradigmas: con base en los resultados de ambos experimentos, argumenten cuándo recomendarían usar TD3 o SAC versus CQL para el sistema de turbinas eólicas real. Consideren explícitamente: disponibilidad de un simulador confiable, costo de interacción con el sistema real, calidad del dataset histórico disponible, y urgencia de despliegue. Esta pregunta no puede responderse correctamente sin los resultados específicos de sus experimentos.


e. Investigación bibliográfica: busquen y lean un paper publicado entre 2023 y 2025 que aplique Offline RL (CQL, BCQ, IQL, o variantes) a un problema de control de sistemas energéticos, robótica industrial, o infraestructura física. El paper debe ser de NeurIPS, ICML, ICLR, IEEE Transactions on Power Systems, o similar. Escriban un resumen técnico de media página con: el problema que resuelve, qué algoritmo de Offline RL usa y por qué, los resultados principales, y una reflexión sobre si las limitaciones del dataset que identificaron en la Task 1.1c también aparecen en ese trabajo y cómo las manejan.
